In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
# Define the file path 
file_path = r"/Causal_Inference/Treatment_effects/data/lalonde.dta"
# Read the Stata file into a pandas DataFrame 
df = pd.read_stata(file_path) 

from scipy.stats import ttest_ind

def perform_ttest(group1, group2):
    t_stat, p_val = ttest_ind(group1, group2, equal_var=False)
    return t_stat, p_val

pre_treatment_vars = ['age', 'educ', 're74', 're75']

treated = df[df['treat'] == 1]
control = df[df['treat'] == 0]

ttest_results = []
for var in pre_treatment_vars:
    t_stat, p_val = perform_ttest(treated[var], control[var])
    ttest_results.append({'Variable': var, 'T-Statistic': round(t_stat, 2), 'P-Value': f"{p_val:.3f}"})

ttest_results_df = pd.DataFrame(ttest_results)

ttest_results_df


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

def perform_ztest(count1, nobs1, count2, nobs2):
    z_stat, p_val = proportions_ztest([count1, count2], [nobs1, nobs2])
    return z_stat, p_val

binary_vars = ['hispan', 'white','married', 'nodegree']

ztest_results = []
for var in binary_vars:
    count_treated = treated[var].sum()
    count_control = control[var].sum()
    nobs_treated = treated[var].count()
    nobs_control = control[var].count()
    z_stat, p_val = perform_ztest(count_treated, nobs_treated, count_control, nobs_control)
    ztest_results.append({'Variable': var, 'Z-Statistic': round(z_stat, 2), 'P-Value': f"{p_val:.3f}"})

ztest_results = pd.DataFrame(ztest_results)

ztest_results



In [ ]:
import statsmodels.formula.api as smf

reg = smf.ols('re78 ~ treat + age + educ + married + nodegree + hispan + white + re74 + re75', data=df)
res = reg.fit()
print(res.summary())

pre_treatment_vars = ['age', 'educ', 'married', 'nodegree', 'hispan', 'white', 're74', 're75']
X = df[pre_treatment_vars]
y = df['treat']

from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X, y)
df['propensity_score_logistic'] = log_reg.predict_proba(X)[:, 1]
df.head()

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=1000)
rf.fit(X, y)
df['propensity_score_random_forest'] = rf.predict_proba(X)[:, 1]
df.head()



In [ ]:
from sklearn.neighbors import NearestNeighbors

treated = df[df['treat'] == 1]
control = df[df['treat'] == 0]

nn = NearestNeighbors(n_neighbors=1)

nn.fit(control[['propensity_score_logistic']])

distances, indices = nn.kneighbors(treated[['propensity_score_logistic']])

matched_control_indices = indices.flatten()
new_control = control.iloc[matched_control_indices]

matched_data = pd.concat([treated, new_control])
type(matched_data)
matched_data.to_stata("/Causal_Inference/Treatment_effects/outputs/matched_data.dta", write_index=False)
matched_data.head()

In [ ]:
def standarized_mean_difference(data, covars):
    
    treated = data[data['treat'] == 1]
    control = data[data['treat'] == 0]
    
    smd = {}
    
    for var in covars:
        mean_treated = treated[var].mean()
        mean_control = control[var].mean()
        sd_treated = treated[var].std()
        sd_control = control[var].std()
        smd[var] = abs(mean_treated - mean_control) / np.sqrt((sd_treated**2 + sd_control**2) / 2)
        
    return smd


smd_before_matching = standarized_mean_difference(df, pre_treatment_vars)
smd_after_matching = standarized_mean_difference(matched_data, pre_treatment_vars)
treated_mean = matched_data[matched_data['treat'] == 1]['re78'].mean()
matched_contol_mean =  matched_data[matched_data['treat'] == 0]['re78'].mean()

treatment_effect = treated_mean - matched_contol_mean
print(treatment_effect)

reg = smf.ols('re78 ~ treat + age + educ + married + nodegree + hispan + white + re74 + re75', data=matched_data)
res = reg.fit()

print(res.summary())


In [ ]:
import datetime
print(f"PSM using lalonde finished")
print(f"Program completed on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")